In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import os
import time
import json
import psutil

# Start the timer for the pure Python analysis notebook
notebook_start_time = time.time()
print("Final Python analysis execution started...")

# Redoing some alanysis to align with SQL analysis

## Data Import 
Importing small sample of data to use as a prototype. Here, we will be using python with the pandas library to attain a performance baseline.

In [ ]:
df = pd.read_csv("trips_prototype.csv")
df.head()

In [ ]:
df["started_at"] = pd.to_datetime(df["started_at"], utc=True)
df["ended_at"] = pd.to_datetime(df["ended_at"], utc=True)
df["ride_length"] = df["ended_at"] - df["started_at"]
df["ride_length_minutes"] = df["ride_length"].dt.total_seconds() / 60
df.head()

## Making Time and Day cols
Allows for us to perform day and time analysis

In [ ]:
df["hour"] = df["started_at"].dt.hour
df["day_of_week"] = df["started_at"].dt.day_name()

df[["started_at", "hour", "day_of_week"]].head()

## Bike type usage
Here, we wanted to find the relationship between bike type and user type. We found the average ride duration for each bike and user combination. This analysis helps identify user preferences for specific bike types and highlights the differences between members and casual riders that we have previously theorised about. 
The shear number of member usage is higher, but when we look at the percentage usage, the figures start to make sense. Members seem to prefer the "classic bike" over the electric more so than the casual users do, this could be down to members being into fitness or simply the bike availability in their residential areas.
The only question we are left with right now is why do only casual users use "docked bikes"?

In [ ]:
bike_user_ct = pd.crosstab(df["rideable_type"], df["member_casual"])
print("Bike type x user type (counts):")
print(bike_user_ct)


print("\nBike type x user type (row %):")
bike_pct = bike_user_ct.div(bike_user_ct.sum(axis=1), axis=0).round(3)
print(bike_pct)

print("\nAverage ride length (minutes) by bike type and user type:")

avg_duration_bike_user = (
    df.groupby(["rideable_type", "member_casual"])["ride_length_minutes"]
    .mean()
    .round(1)
    .unstack()
)
print(avg_duration_bike_user)

print("\nInterpretation by bike type:")
for bike in bike_user_ct.index:
    casual_share = bike_pct.loc[bike, "casual"] * 100
    member_share = bike_pct.loc[bike, "member"] * 100
    print(f"- {bike}: {casual_share:.1f}% casual, {member_share:.1f}% member")

## Station-based insights
We wanted to find out which stations had the highest numbers of users arriving and departing from them, we can see that "Dearborn St and Erie St" appears one of the focal points, seeing as it is the most popular destination and second most popular starting point. Similar observations can be made for each other station.
We also computed the average ride duration by start station an user type. We filtered this to include stations with 30+ trips to ensure reliability.
Using this information, we can detarmine what areas are residential and which are tourist hotspots among many other things.

In [ ]:
for user_type in ["member", "casual"]:
    print(f"\nTop 10 start stations for {user_type}s:")
    top_start = (
        df[df["member_casual"] == user_type]["start_station_name"]
        .value_counts()
        .head(10)
    )
    print(top_start)

    print(f"\nTop 10 end stations for {user_type}s:")
    top_end = (
        df[df["member_casual"] == user_type]["end_station_name"]
        .value_counts()
        .head(10)
    )
    print(top_end)

min_trips_per_station = 30

avg_duration_station = (
    df.groupby(["start_station_name", "member_casual"])["ride_length_minutes"]
    .agg(["count", "mean"])
    .reset_index()
)



avg_duration_station = avg_duration_station[avg_duration_station["count"] >= min_trips_per_station]

print("\nAverage ride length (minutes) by start station and user type (>= 30 trips):")
print(avg_duration_station.sort_values("mean", ascending=False).head(20))

## Time of day x day of week bike usage
We could use seaborn to display this information on a heatmap as it is very suited for that, but seeing as we are attempting to set a base line for time, we thought it best to stick with text outputs in case graphics take a significantly long amount of time to display.
Now bike usage by day of week and time is legible. The huge numbers of users visible during the work rush hours on weekdays confirms our earlier hypotheses. The low numbers of rides over the weekend do so as well, likely being mostly tourist use.

In [ ]:
#this is basically a text heat map, I wanted to make something easily duplicated in other languages without importing libraries
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
cat_type = pd.CategoricalDtype(categories=day_order, ordered=True)
df["day_of_week"] = df["day_of_week"].astype(cat_type)

for user_type in ["member", "casual"]:
    subset = df[df["member_casual"] == user_type]
    pivot = subset.pivot_table(
        index="day_of_week",
        columns="hour",
        values="ride_id",
        aggfunc="count",
        fill_value=0,
    )

    print(f"\nTrips table (day x hour) for {user_type}:")
    print(pivot)

    print(f"\nTop 5 day-hour cells for {user_type}:")
    top_cells = (
        pivot.stack()
        .sort_values(ascending=False)
        .head(5)
        .reset_index(name="trips")
    )
    for _, row in top_cells.iterrows():
        print(f"- {row['day_of_week']} @ {row['hour']}: {row['trips']} trips")

## Analysing Daily Trends
we can see saturday is extra busy for casual users which supports the theory they are tourists trying to view the city

In [ ]:
weekday_counts = (
    df.groupby(["day_of_week", "member_casual"])
    .size()
    .unstack(fill_value=0)
    .reindex(day_order)
)

member_counts = weekday_counts["member"]
casual_counts = weekday_counts["casual"]


print("Trips by day of week and user type (counts):")
print(weekday_counts)

print("\nDay-of-week share of trips by user type (row %):")
weekday_pct = weekday_counts.div(weekday_counts.sum(axis=1), axis=0).round(3)
print(weekday_pct)

print("\nKey weekday patterns:")

for day in weekday_counts.index:
    casual = weekday_counts.loc[day, "casual"]
    member = weekday_counts.loc[day, "member"]
    print(f"- {day}: {casual} casual vs {member} member trips")

## Busy Stations
We can see the busiest stations, ranked on the number of arrivals + departures

In [ ]:
departures = df.groupby("start_station_name").size().rename("departures")
arrivals = df.groupby("end_station_name").size().rename("arrivals")


station_activity = (
    departures.to_frame()
    .join(arrivals.to_frame(), how="outer")
    .fillna(0)
)

station_activity["total_activity"] = (
    station_activity["departures"] + station_activity["arrivals"]
)



top_stations = station_activity.sort_values("total_activity", ascending=False).head(10)

print("Top stations by total activity (departures + arrivals):")
print(top_stations)

## Ride lengths by bike type
We can see what the average ride length is for both casuals and members for each type of bike.

In [ ]:

valid = df[df["ride_length_minutes"].notna()].copy()

avg_duration_bike_user = (valid.groupby(["rideable_type", "member_casual"])["ride_length_minutes"].mean().round(1).unstack())

print("Average ride length (minutes) by bike type and user type:")
print(avg_duration_bike_user)


## TOTAL TIME AND MEMORY

In the last cell below, we calculate the total time and memory this notebook took to process our prototype data with sql.

In [ ]:
notebook_end_time = time.time()
total_seconds = notebook_end_time - notebook_start_time
minutes = int(total_seconds // 60)
seconds = int(total_seconds % 60)

print("-" * 40)
print(f"Total Python Analysis Execution Time: {minutes} minutes and {seconds} seconds.")

process = psutil.Process(os.getpid())
memory_info = process.memory_info()
memory_usage_mb = memory_info.rss / (1024 * 1024)

print(f"Memory Usage: {memory_usage_mb:.2f} MB")
print("-" * 40)

current_dir = os.getcwd()

if os.path.exists(os.path.join(current_dir, "results")):
    results_dir = os.path.join(current_dir, "results")
else:
    project_root = os.path.abspath(os.path.join(current_dir, ".."))
    results_dir = os.path.join(project_root, "results")

os.makedirs(results_dir, exist_ok=True)
metrics_file_path = os.path.join(results_dir, "metrics.json")

if os.path.exists(metrics_file_path):
    with open(metrics_file_path, "r") as f:
        try:
            metrics_data = json.load(f)
        except json.JSONDecodeError:
            metrics_data = {} 
else:
    metrics_data = {}

metrics_data["python_analysis"] = {
    "total_execution_time_seconds": round(total_seconds, 2),
    "total_execution_time_formatted": f"{minutes}m {seconds}s",
    "memory_usage_mb": round(memory_usage_mb, 2)
}

with open(metrics_file_path, "w") as f:
    json.dump(metrics_data, f, indent=4)

print(f"Metrics successfully updated in: {metrics_file_path}")